# 02 - Convert rasters to Cloud-Optimized GeoTIFF (COG)

In a nutshell, a COG is a regular GeoTIFF, with:

* **blocks** (not stripes), so that a region can be read with a few HTTP range requests;
* **compression** of the blocks;
* **overviews** (reduced-resolution copies), so that zoomed-out views are cheap;
* all metadata (header and block offsets) at the **beginning of the file**, followed by overviews and full-resolution data.

COGs work with virtually all (legacy) GIS software, since it they are regular GeoTIFFs. They are established for web visualization (e.g. in the Earth Observation community).

Let's try out different COG configurations, testing different:

* **Block size**: the size of the blocks sets the granularity at which the COG can be accessed with HTTP requests. Too small means many requests are needed in order to access data (and latency, not throughput, is often the bottleneck). Too large "wastes" bytes when only a small area is needed. For tiled datasets, also consider values that are commensurate with the tile size (useful for building virtual mosaics).
* **Compression**: can make a big difference in file size. Elements to consider: 
  * Is lossy compression acceptable (e.g. JPEG, WEBP, LERC with max error)?
  * Need universal (legacy) GIS-software support? Then use LZW/DEFLATE.
  * If supported, consider ZSTD (same compression ratio as DEFLATE, faster decompression).

We generate COGs via the GDAL's [COG driver](https://gdal.org/en/stable/drivers/raster/cog.html), using the [rasterio](https://rasterio.readthedocs.io) Python library.


In [1]:
import pathlib
import time

import fsspec
import pandas as pd
import rasterio.shutil

from utils import BUCKET_URI, ENDPOINT_URL, USER_PREFIX_URI, gdal_env, print_raster_info, vsis3, https_url

In [2]:
fs = fsspec.filesystem("s3", endpoint_url=ENDPOINT_URL)

def convert(src: str, dst: str, **options) -> dict:
    """Convert `src` to a COG with the given creation options, output to `dst`."""
    t0 = time.perf_counter()
    with gdal_env(CPL_VSIL_USE_TEMP_FILE_FOR_RANDOM_WRITE="YES"):
        rasterio.shutil.copy(vsis3(src), vsis3(dst), driver="COG", **options)
    elapsed = time.perf_counter() - t0
    size = fs.size(dst)
    return {
        "src": src,
        "dst": dst,
        "time_s": round(elapsed, 2),
        "size_MB": round(size / 1e6, 2),
        **options,
    }

## Digital Surface Model

We first compare several variants on a single tile.

In [3]:
dsm_tile_uri = f"{BUCKET_URI}/AHN-4/DSM05/AHN4_R_136000_455000.TIF"
print(f"Size of original: {fs.size(dsm_tile_uri) / 1e6:.2f} MB")

Size of original: 9.79 MB


In [4]:
DSM_VARIANTS = {
    "256-DEFLATE": dict(BLOCKSIZE=256, COMPRESS="DEFLATE"),
    "512-DEFLATE": dict(BLOCKSIZE=512, COMPRESS="DEFLATE"),
    "1024-DEFLATE": dict(BLOCKSIZE=1024, COMPRESS="DEFLATE"),
    "512-LZW": dict(BLOCKSIZE=512, COMPRESS="LZW"),
    "512-ZSTD": dict(BLOCKSIZE=512, COMPRESS="ZSTD"),
    "512-LERC_ZSTD-1cm": dict(BLOCKSIZE=512, COMPRESS="LERC_ZSTD", MAX_Z_ERROR=0.01),  # lossy!
}

In [5]:
dsm_results = [
    {"variant": name, **convert(dsm_tile_uri, f"{USER_PREFIX_URI}/AHN-4/DSM05/variants/{name}.tif", **options)}
    for name, options in DSM_VARIANTS.items()
]
dsm_results_df = pd.DataFrame(dsm_results)
dsm_results_df.drop(columns=["src", "dst"])

,variant,time_s,size_MB,BLOCKSIZE,COMPRESS,MAX_Z_ERROR
0,256-DEFLATE,5.10,13.87,256,DEFLATE,NaN
1,512-DEFLATE,3.77,13.87,512,DEFLATE,NaN
2,1024-DEFLATE,3.69,13.26,1024,DEFLATE,NaN
3,512-LZW,4.49,17.25,512,LZW,NaN
4,512-ZSTD,3.96,13.90,512,ZSTD,NaN
5,512-LERC_ZSTD-1cm,1.98,5.95,512,LERC_ZSTD,0.01


> ## Challenge
> * Why are most COGs larger than the original file?
> * Which compression gives the smallest file? Which one is the fastest to write?
> * Try adding a variant of your own. For instance, `OVERVIEWS="NONE"`, `LEVEL=9`, or a different `MAX_Z_ERROR`. See the [COG driver docs](https://gdal.org/en/stable/drivers/raster/cog.html#creation-options) for all options.

Let's explore one of the tile:

In [6]:
dsm_cog_uri = f"{USER_PREFIX_URI}/AHN-4/DSM05/variants/512-ZSTD.tif"
print_raster_info(dsm_cog_uri)

URL:          s3://cng-forum-workshop/users/fnattino/AHN-4/DSM05/variants/512-ZSTD.tif
Driver:       GTiff
Size:         2000 x 2000 , Bands: 1
CRS:          EPSG:7415
Bounds:       BoundingBox(left=136000.0, bottom=455000.0, right=137000.0, top=456000.0)
Resolution:   (0.5, 0.5)
Dtypes:       ('float32',)
Nodata:       -9999.0
Block shapes: [(512, 512)]
Tiled:        True
Compression:  ZSTD
Layout:       COG
Bands:       
 * Band 1: description=Hoogte (m NAP), overviews=[2, 4]


Let's also visualize it with the [source cooperative COG web viewer](https://github.com/source-cooperative/cog-viewer):

In [7]:
print(f"Visualize at: https://source-cooperative.github.io/cog-viewer/?url={https_url(dsm_cog_uri)}")

Visualize at: https://source-cooperative.github.io/cog-viewer/?url=https://objectstore.surf.nl/cng-forum-workshop/users/fnattino/AHN-4/DSM05/variants/512-ZSTD.tif


### Convert a few tiles

Pick the options you prefer and convert a 2 x 2 km area (4 tiles). We will catalog these files in notebook 05.

In [8]:
dsm_options = DSM_VARIANTS["512-ZSTD"]  # <- your choice here
dsm_tiles = [
    f"{BUCKET_URI}/AHN-4/DSM05/AHN4_R_{x}_{y}.TIF"
    for x in (135000, 136000)
    for y in (454000, 455000)
]
for tile in dsm_tiles:
    dst = f"{USER_PREFIX_URI}/AHN-4/DSM05/cog/{pathlib.PurePath(tile).stem}.tif"
    print(convert(tile, dst, **dsm_options))

{'src': 's3://cng-forum-workshop/AHN-4/DSM05/AHN4_R_135000_454000.TIF', 'dst': 's3://cng-forum-workshop/users/fnattino/AHN-4/DSM05/cog/AHN4_R_135000_454000.tif', 'time_s': 4.63, 'size_MB': 12.36, 'BLOCKSIZE': 512, 'COMPRESS': 'ZSTD'}
{'src': 's3://cng-forum-workshop/AHN-4/DSM05/AHN4_R_135000_455000.TIF', 'dst': 's3://cng-forum-workshop/users/fnattino/AHN-4/DSM05/cog/AHN4_R_135000_455000.tif', 'time_s': 4.22, 'size_MB': 13.01, 'BLOCKSIZE': 512, 'COMPRESS': 'ZSTD'}
{'src': 's3://cng-forum-workshop/AHN-4/DSM05/AHN4_R_136000_454000.TIF', 'dst': 's3://cng-forum-workshop/users/fnattino/AHN-4/DSM05/cog/AHN4_R_136000_454000.tif', 'time_s': 4.55, 'size_MB': 13.42, 'BLOCKSIZE': 512, 'COMPRESS': 'ZSTD'}
{'src': 's3://cng-forum-workshop/AHN-4/DSM05/AHN4_R_136000_455000.TIF', 'dst': 's3://cng-forum-workshop/users/fnattino/AHN-4/DSM05/cog/AHN4_R_136000_455000.tif', 'time_s': 4.44, 'size_MB': 13.9, 'BLOCKSIZE': 512, 'COMPRESS': 'ZSTD'}


## RGB Aerial Images

Here lossy compression is common practice. Tiles are large (12500 x 12500 pixels), so conversions take a few seconds each.

In [9]:
rgb_tile_uri = f"{BUCKET_URI}/aerial-images-NL-2024/RGB/2024_136000_455000_RGB_hrl.tif"
print(f"Size of original: {fs.size(rgb_tile_uri) / 1e6:.2f} MB")

Size of original: 28.90 MB


In [10]:
RGB_VARIANTS = {
    "512-JPEG-q75": dict(BLOCKSIZE=512, COMPRESS="JPEG", QUALITY=75),
    "512-JPEG-q90": dict(BLOCKSIZE=512, COMPRESS="JPEG", QUALITY=90),
    "512-WEBP-q75": dict(BLOCKSIZE=512, COMPRESS="WEBP", QUALITY=75),
    "512-ZSTD": dict(BLOCKSIZE=512, COMPRESS="ZSTD"),  # lossless
}

In [11]:
rgb_results = [
    {"variant": name, **convert(rgb_tile_uri, f"{USER_PREFIX_URI}/aerial-images-NL-2024/RGB/variants/{name}.tif", **options)}
    for name, options in RGB_VARIANTS.items()
]
rgb_results_df = pd.DataFrame(rgb_results)
rgb_results_df.drop(columns=["src", "dst"])

,variant,time_s,size_MB,BLOCKSIZE,COMPRESS,QUALITY
0,512-JPEG-q75,15.97,23.37,512,JPEG,75.0
1,512-JPEG-q90,20.31,43.17,512,JPEG,90.0
2,512-WEBP-q75,24.41,12.15,512,WEBP,75.0
3,512-ZSTD,97.67,330.03,512,ZSTD,NaN


> ## Challenge
> * How much does the lossless variant cost in terms of storage?

Again, let's explore one tile:

In [12]:
rgb_cog_uri = f"{USER_PREFIX_URI}/aerial-images-NL-2024/RGB/variants/512-JPEG-q75.tif"
print_raster_info(rgb_cog_uri)

URL:          s3://cng-forum-workshop/users/fnattino/aerial-images-NL-2024/RGB/variants/512-JPEG-q75.tif
Driver:       GTiff
Size:         12500 x 12500 , Bands: 3
CRS:          EPSG:28992
Bounds:       BoundingBox(left=136000.0, bottom=454000.0, right=137000.0, top=455000.0)
Resolution:   (0.08, 0.08)
Dtypes:       ('uint8', 'uint8', 'uint8')
Nodata:       None
Block shapes: [(512, 512), (512, 512), (512, 512)]
Tiled:        True
Compression:  JPEG
Layout:       COG
Bands:       
 * Band 1: description=None, overviews=[2, 4, 8, 16, 32]
 * Band 2: description=None, overviews=[2, 4, 8, 16, 32]
 * Band 3: description=None, overviews=[2, 4, 8, 16, 32]


In [13]:
print(f"Visualize at: https://source-cooperative.github.io/cog-viewer/?url={https_url(rgb_cog_uri)}")

Visualize at: https://source-cooperative.github.io/cog-viewer/?url=https://objectstore.surf.nl/cng-forum-workshop/users/fnattino/aerial-images-NL-2024/RGB/variants/512-JPEG-q75.tif


### Convert a few tiles

Again, pick the options you prefer and convert 4 tiles: 

In [14]:
rgb_options = RGB_VARIANTS["512-JPEG-q75"]  # <- your choice here
rgb_tiles = [
    f"{BUCKET_URI}/aerial-images-NL-2024/RGB/2024_{x}_{y}_RGB_hrl.tif"
    for x in (135000, 136000)
    for y in (455000, 456000)
]
for tile in rgb_tiles:
    dst = f"{USER_PREFIX_URI}/aerial-images-NL-2024/RGB/cog/{pathlib.PurePath(tile).stem}.tif"
    print(convert(tile, dst, **rgb_options))

{'src': 's3://cng-forum-workshop/aerial-images-NL-2024/RGB/2024_135000_455000_RGB_hrl.tif', 'dst': 's3://cng-forum-workshop/users/fnattino/aerial-images-NL-2024/RGB/cog/2024_135000_455000_RGB_hrl.tif', 'time_s': 15.67, 'size_MB': 19.1, 'BLOCKSIZE': 512, 'COMPRESS': 'JPEG', 'QUALITY': 75}
{'src': 's3://cng-forum-workshop/aerial-images-NL-2024/RGB/2024_135000_456000_RGB_hrl.tif', 'dst': 's3://cng-forum-workshop/users/fnattino/aerial-images-NL-2024/RGB/cog/2024_135000_456000_RGB_hrl.tif', 'time_s': 15.62, 'size_MB': 18.68, 'BLOCKSIZE': 512, 'COMPRESS': 'JPEG', 'QUALITY': 75}
{'src': 's3://cng-forum-workshop/aerial-images-NL-2024/RGB/2024_136000_455000_RGB_hrl.tif', 'dst': 's3://cng-forum-workshop/users/fnattino/aerial-images-NL-2024/RGB/cog/2024_136000_455000_RGB_hrl.tif', 'time_s': 16.85, 'size_MB': 23.37, 'BLOCKSIZE': 512, 'COMPRESS': 'JPEG', 'QUALITY': 75}
{'src': 's3://cng-forum-workshop/aerial-images-NL-2024/RGB/2024_136000_456000_RGB_hrl.tif', 'dst': 's3://cng-forum-workshop/users/f